In [0]:
import datetime
import uuid
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

def init_logging_table(spark: SparkSession, table_name: str):
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            run_id STRING,
            layer STRING,
            load_type STRING,
            target_table STRING,
            parameter STRING,
            start_time TIMESTAMP,
            end_time TIMESTAMP,
            status STRING,
            rows_inserted LONG,
            rows_updated LONG,
            rows_quarantined LONG,
            error_message STRING,
            load_timestamp TIMESTAMP
        ) USING DELTA;
    """)

def log_execution(spark: SparkSession, table_name: str, layer: str, load_type: str, 
                  target_table: str, param: str, start_time: datetime.datetime, 
                  end_time: datetime.datetime, status: str, inserted: int = 0, 
                  updated: int = 0, quarantined: int = 0, error: str = None):
    
    run_id = str(uuid.uuid4())
    log_data = [(
        run_id, layer, load_type, target_table, param,
        start_time, end_time, status, int(inserted),
        int(updated), int(quarantined), error, datetime.datetime.utcnow()
    )]
    
    schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("load_type", StringType(), False),
        StructField("target_table", StringType(), False),
        StructField("parameter", StringType(), True),
        StructField("start_time", TimestampType(), False),
        StructField("end_time", TimestampType(), False),
        StructField("status", StringType(), False),
        StructField("rows_inserted", LongType(), False),
        StructField("rows_updated", LongType(), False),
        StructField("rows_quarantined", LongType(), False),
        StructField("error_message", StringType(), True),
        StructField("load_timestamp", TimestampType(), False)
    ])
    
    df = spark.createDataFrame(log_data, schema)
    df.write.format("delta").mode("append").saveAsTable(table_name)

def get_last_merge_metrics(spark: SparkSession, target_table: str):
    try:
        metrics = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").collect()[0]["operationMetrics"]
        inserted = int(metrics.get("numTargetRowsInserted", 0))
        updated = int(metrics.get("numTargetRowsUpdated", 0))
        return inserted, updated
    except Exception:
        return 0, 0

def run_step(spark: SparkSession, log_table: str, layer: str, load_type: str, 
             target_table: str, param: str, step_fn):
    start_time = datetime.datetime.utcnow()
    try:
        inserted, updated, quarantined = step_fn()
        end_time = datetime.datetime.utcnow()
        log_execution(spark, log_table, layer, load_type, target_table, param, 
                      start_time, end_time, "SUCCESS", inserted, updated, quarantined)
        print(f"[{layer}] Successfully processed {param}")
    except Exception as e:
        end_time = datetime.datetime.utcnow()
        log_execution(spark, log_table, layer, load_type, target_table, param, 
                      start_time, end_time, "FAILED", 0, 0, 0, str(e))
        print(f"[{layer}] Failed on {param}: {str(e)}")